In [1]:
import pandas as pd
import numpy as np
import csv

# File upload

In [2]:
file_path = "ergebnisse_nummeriert.csv"

with open(file_path, "r", encoding="utf-8-sig") as f:
    sample = f.read(5000)
    dialect = csv.Sniffer().sniff(sample)

df = pd.read_csv(
    file_path,
    sep=dialect.delimiter,
    encoding="utf-8-sig",
    engine="python",
    on_bad_lines="warn"
)

df.head()

,Id,Startzeit,Fertigstellungszeit,E-Mail,Name,Your role,Work Experience,Familiarity.Familiarity with Machine Learning,Familiarity.Familiarity with Credit Risk Models,Familiarity.Familiarity with Python,...,Unnamed: 157,Unnamed: 158,Unnamed: 159,Unnamed: 160,Unnamed: 161,Unnamed: 162,Unnamed: 163,Unnamed: 164,Unnamed: 165,Unnamed: 166
0,1,09.06.2026 08:43,09.06.2026 10:55,anonymous,NaN,Risk Management,2–5 years,4,7,7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,09.06.2026 18:48,09.06.2026 19:46,anonymous,NaN,Data Science / Analytics,2–5 years,5,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,10.06.2026 12:29,10.06.2026 17:33,anonymous,NaN,Data Science / Analytics,< 2 years,5,1,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,10.06.2026 19:57,10.06.2026 20:55,anonymous,NaN,Data Science / Analytics,6–10 years,5,1,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,10.06.2026 18:41,10.06.2026 21:30,anonymous,NaN,Data Science / Analytics,< 2 years,2,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Calculations of Dimensions

In [ ]:
levels = [0, 1, 2, 3, 4]

dimensions = {
    "Interpretability": "Interpretability",
    "Regulatory Plausibility": "Regulatory Plausibility",
    "Trust": "Trust",
    "Acceptance": "Acceptance"
}


def get_block_columns(df, start_idx, block_size=13):
    return df.columns[start_idx:start_idx + block_size]


# only likert scales
likert_cols = [
    c for c in df.columns
    if "Block A" in c
    or "Block B" in c
    or "Block C" in c
    or "Block D" in c
]

print("Number of Likert columns:", len(likert_cols))

#scores

def compute_scores(likert_cols, use_case_name, offset):
    rows = []

    for level in levels:
        # 13 questions per level
        # 4 Interpretability, 4 Regulatory, 3 Trust, 2 Acceptance
        level_cols = likert_cols[(offset + level) * 13 : (offset + level + 1) * 13]

        row = {"Level": f"Level {level}"}

        row["Interpretability"] = df[level_cols[0:4]].mean(axis=1).mean()
        row["Regulatory Plausibility"] = df[level_cols[4:8]].mean(axis=1).mean()
        row["Trust"] = df[level_cols[8:11]].mean(axis=1).mean()
        row["Acceptance"] = df[level_cols[11:13]].mean(axis=1).mean()

        rows.append(row)

    out = pd.DataFrame(rows)
    out.iloc[:, 1:] = out.iloc[:, 1:].round(2)
    return out


pd_scores = compute_scores(likert_cols, "PD", offset=0)
marketing_scores = compute_scores(likert_cols, "Marketing", offset=5)

print("PD Scores")
display(pd_scores)

print("Marketing Scores")
display(marketing_scores)

Number of Likert columns: 130
PD Scores


,Level,Interpretability,Regulatory Plausibility,Trust,Acceptance
0,Level 0,3.33,3.19,3.33,3.28
1,Level 1,3.25,3.00,3.07,2.72
2,Level 2,3.39,3.14,3.07,2.89
3,Level 3,3.75,3.61,3.41,3.17
4,Level 4,4.08,3.94,3.96,3.83


Marketing Scores


,Level,Interpretability,Regulatory Plausibility,Trust,Acceptance
0,Level 0,3.03,3.19,3.11,3.06
1,Level 1,2.69,3.00,2.93,2.83
2,Level 2,3.06,2.97,3.00,3.17
3,Level 3,3.17,3.19,3.11,3.06
4,Level 4,3.56,3.56,3.30,3.17
